In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("Accelerometer_XYZ_Combined.csv")

df.head()

,time,x,y,z
0,2026-09-07T09:35:51.346000Z,0.191426,7.548757,6.999604
1,2026-09-07T09:35:52.417000Z,0.064008,6.747760,6.534799
2,2026-09-07T09:35:53.363000Z,0.037687,7.794022,5.891129
3,2026-09-07T09:35:54.373000Z,0.166899,7.875976,5.965905
4,2026-09-07T09:35:55.382000Z,0.238086,7.718050,5.871388


In [3]:
df = df.dropna(subset=["x", "y", "z"]).copy()

df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)

In [4]:
df["magnitude"] = np.sqrt(
    df["x"]**2 + df["y"]**2 + df["z"]**2
)

In [5]:
df["window"] = df["time"].dt.floor("10s")

window_stats = df.groupby("window").agg(
    mean_magnitude=("magnitude", "mean"),
    std_magnitude=("magnitude", "std"),
    max_magnitude=("magnitude", "max"),
    min_magnitude=("magnitude", "min")
).reset_index()

In [6]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

features = window_stats[
    ["mean_magnitude", "std_magnitude",
     "max_magnitude", "min_magnitude"]
].copy()

features = features.dropna()

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

window_stats.loc[features.index, "cluster"] = (
    kmeans.fit_predict(features_scaled)
)

C:\Users\ruben\anaconda3\anaconda\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\ruben\anaconda3\anaconda\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\ruben\anaconda3\anaconda\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ruben\anaconda3\anaconda\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~

In [7]:
cluster_summary = window_stats.groupby("cluster").agg(
    windows=("cluster", "count"),
    average_variability=("std_magnitude", "mean"),
    average_magnitude=("mean_magnitude", "mean"),
    average_max=("max_magnitude", "mean")
).sort_values("average_variability")

print(cluster_summary)

         windows  average_variability  average_magnitude  average_max
cluster                                                              
0.0          169             0.367141           9.813745    10.454871
1.0           60             1.322873           9.554134    11.800214
2.0           63             1.535229          10.389396    13.873292


In [8]:
cluster_order = cluster_summary.index.tolist()

label_mapping = {
    cluster_order[0]: 0,
    cluster_order[1]: 1,
    cluster_order[2]: 2
}

window_stats["activity"] = window_stats["cluster"].map(label_mapping)

print(window_stats[["window", "std_magnitude", "activity"]].head(20))

                      window  std_magnitude  activity
0  2026-09-07 09:35:50+00:00       0.232542         0
1  2026-09-07 09:36:00+00:00       0.580502         0
2  2026-09-07 09:36:10+00:00       0.169701         0
3  2026-09-07 09:36:20+00:00       0.005861         0
4  2026-09-07 09:36:30+00:00       0.014268         0
5  2026-09-07 09:36:40+00:00       0.359650         0
6  2026-09-07 09:36:50+00:00       0.175640         0
7  2026-09-07 09:37:00+00:00       0.176256         0
8  2026-09-07 09:37:10+00:00       0.332466         0
9  2026-09-07 09:37:20+00:00       0.061757         0
10 2026-09-07 09:37:30+00:00       0.400858         0
11 2026-09-07 09:37:40+00:00       0.059008         0
12 2026-09-07 09:37:50+00:00       1.395156         2
13 2026-09-07 09:38:00+00:00       0.005987         0
14 2026-09-07 09:38:10+00:00       0.011313         0
15 2026-09-07 09:38:20+00:00       0.039956         0
16 2026-09-07 09:38:50+00:00       0.183678         0
17 2026-09-07 09:39:00+00:00

In [9]:
print(window_stats.columns.tolist())
print(window_stats["activity"].value_counts().sort_index())

['window', 'mean_magnitude', 'std_magnitude', 'max_magnitude', 'min_magnitude', 'cluster', 'activity']
activity
0    169
1     60
2     63
Name: count, dtype: int64


In [10]:
df = df.merge(
    window_stats[["window", "activity"]],
    on="window",
    how="left"
)

print(df.columns.tolist())

['time', 'x', 'y', 'z', 'magnitude', 'window', 'activity']


In [11]:
activity_summary = df.groupby("activity").agg(
    mean_magnitude=("magnitude", "mean"),
    std_magnitude=("magnitude", "std"),
    min_magnitude=("magnitude", "min"),
    max_magnitude=("magnitude", "max"),
    samples=("magnitude", "count")
)

activity_summary

,mean_magnitude,std_magnitude,min_magnitude,max_magnitude,samples
activity,,,,,
0,9.825153,0.500827,7.925451,12.466179,3095
1,9.553048,1.372711,3.957571,14.690068,1186
2,10.388553,1.682416,5.807916,25.641870,1231


In [12]:
annotation = window_stats[["window", "activity"]].copy()

annotation["filename"] = annotation["window"].dt.strftime(
    "%Y%m%d%H%M%S"
)

annotation["activity"] = annotation["activity"].astype(int)

annotation = annotation[["filename", "activity"]]

annotation.to_csv("annotation.csv", index=False)